# Evaluating Classification Models

## Credit Card Fraud Data

In [ ]:
import pandas as pd

df_fraud = pd.read_csv(
    "https://datasci112.stanford.edu/data/fraud.csv")
df_fraud

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer

pipeline = make_pipeline(
    make_column_transformer(
        (OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         ["card4", "card6", "P_emaildomain"]),
        remainder="passthrough"),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5))

In [ ]:
X_train = df_fraud.drop("isFraud", axis="columns")
y_train = df_fraud["isFraud"]

In [ ]:
from sklearn.model_selection import cross_val_score

cross_val_score(
    pipeline,
    X=X_train, y=y_train,
    scoring="accuracy",
    cv=10
).mean()

In [ ]:
y_train.value_counts()

## Precision and Recall

We can determine the (training) precision and recall for each class from the confusion matrix.

In [ ]:
from sklearn.metrics import confusion_matrix

pipeline.fit(X_train, y_train)
y_train_ = pipeline.predict(X_train)
confusion_matrix(y_train, y_train_)

For example, the precision and recall for fraudulent transactions are:

In [ ]:
precision, recall = 595 / (595 + 118), 595 / (595 + 1524)
precision, recall

And the F1-score for fraudulent transactions is the harmonic mean of these two numbers.

In [ ]:
2 / (1 / precision + 1 / recall)

There is a precision, recall, and F1 score for each class. But Scikit-Learn requires that the `scoring=` parameter be a single number.

For this, we can use

- "precision_macro"
- "recall_macro"
- "f1_macro"

which average the score over the classes.

In [ ]:
cross_val_score(
    pipeline,
    X=X_train, y=y_train,
    scoring="f1_macro",
    cv=10
).mean()

Precision-recall curves show the tradeoff between precision and recall.

In [ ]:
y_train_probs_ = pipeline.predict_proba(X_train)
y_train_probs_

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

PrecisionRecallDisplay.from_estimator(
    pipeline, X_train, y_train, plot_chance_level=True
)